## Ingestion notebook that ingests data from external API source

### Earthquake API
-   source: "https://earthquake.usgs.gov"
-   port: "443" 
-   base path: "/earthqakes/feed/v1.0/"
-   URL: "/summary/all_day.geojson"

In [0]:
  # full URL path
# https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson

In [0]:
import requests
import json
import datetime


In [0]:
# Access the connections we created during setup

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

conn = w.connections.get(name="earthquake_api_http_connection")
base_url = f"{conn.options['host']}{conn.options['base_path']}"

In [0]:
base_url

In [0]:
# you will see this at the top of screen, and there you can change the value to whatever you want and it will update here
dbutils.widgets.text("api_source", "summary/all_day.geojson")
api_source = dbutils.widgets.get("api_source") # set it to a variable to use it in next block. Never have to change it now, only update the widget and can also be used as a parameter when running the task job(Parameters at runtime)

In [0]:
# This is the code to get the data from API
# Every time we run this, it puts a new file in the volume
url = f"{base_url}" + api_source
response = requests.get(url)
if response.status_code != 200:
    raise Exception(f"Request failed with status code {response.status_code}")
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
print("Current Date: ", current_date)
dbutils.fs.put(f"/Volumes/lakehouse/01_raw/raw/earthquake_data_{current_date}.json", json.dumps(data), True)
#print(url)